<a href="https://colab.research.google.com/github/Manarafm/privacy-preserving-HE/blob/main/PIMA_NEW_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q uv
!uv venv --clear --python 3.11 /content/cml
!uv pip install --python /content/cml/bin/python concrete-ml pandas scikit-learn scipy psutil
!/content/cml/bin/python -c "import concrete.ml.version as v; print('Concrete ML', v.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 68.3 MB/s eta 0:00:00
Using CPython 3.11.17
Creating virtual environment at: cml
Activate with: source cml/bin/activate
Using Python 3.11.17 environment at: cml
Resolved 72 packages in 2.19s
Prepared 72 packages in 1m 56s
Installed 72 packages in 1.43s
 + anyio==4.15.1
 + brevitas==0.10.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + coloredlogs==15.0.1
 + concrete-ml==1.9.0
 + concrete-ml-extensions==0.1.9
 + concrete-python==2.10.0
 + dependencies==2.0.1
 + dill==0.4.1
 + filelock==4.1.1
 + flatbuffers==25.12.19
 + fsspec==2026.9.0
 + h11==0.16.0
 + hf-xet==1.7.0
 + httpcore2==2.13.1
 + httpx2==2.13.1
 + huggingface-hub==2.2.0
 + humanfriendly==10.0
 + hummingbird-ml==0.4.11
 + idna==3.20
 + importlib-resources==7.1.0
 + jinja2==3.1.6
 + joblib==1.6.0
 + jsonpickle==4.1.3
 + markupsafe==3.0.4
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==1.26.4
 + nvidia-cublas-cu12==12.1.3.1
 + nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-nvrtc-

In [2]:
import pandas as pd
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
pima = pd.read_csv(url, header=None,
    names=['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigree','Age','Outcome'])
pima.to_csv("pima.csv", index=False)
print(pima.shape)

(768, 9)


In [3]:
%%writefile eval_utility.py
import numpy as np, pandas as pd, time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression as SkLR
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from concrete.ml.sklearn import LogisticRegression as ConcreteLR

# ---- settings (edit these) ----
CSV, TARGET = "pima.csv", "Outcome"
K = 5                      # number of hospitals
N_SEEDS = 10               # number of repeated random splits
N_BITS_LIST = [6, 8]       # quantization bit-widths to test

df = pd.read_csv(CSV)
y_all = df[TARGET].astype(int).values
for c in ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]:
    if c in df.columns:
        df[c] = df[c].replace(0, np.nan)          # PIMA: 0 means missing
X_all = df.drop(columns=[TARGET]).astype(float).values
col = lambda p: p[:, 1] if p.ndim == 2 else p

def metrics(y, p, p_ref):
    pred = (p >= 0.5).astype(int)
    return dict(
        accuracy=accuracy_score(y, pred),
        precision=precision_score(y, pred, zero_division=0),
        recall=recall_score(y, pred, zero_division=0),
        f1=f1_score(y, pred, zero_division=0),
        error_rel=np.linalg.norm(p - p_ref) / np.linalg.norm(p_ref),   # Eq. 10: output difference vs plaintext
        mse=np.mean((y - p) ** 2),                                      # Eq. 11 on probabilities (Brier score)
    )

rows = []
for seed in range(N_SEEDS):
    Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.25, stratify=y_all, random_state=seed)
    imp = SimpleImputer(strategy="median").fit(Xtr)
    sc = StandardScaler().fit(imp.transform(Xtr))
    Xtr, Xte = sc.transform(imp.transform(Xtr)), sc.transform(imp.transform(Xte))

    # hospitals train locally, server averages (FedAvg, weighted by size)
    parts = np.array_split(np.random.default_rng(seed).permutation(len(ytr)), K)
    loc = [(len(i), *(lambda m: (m.coef_[0], m.intercept_[0]))(SkLR(max_iter=2000).fit(Xtr[i], ytr[i]))) for i in parts]
    N = sum(n for n, _, _ in loc)
    sk = SkLR().fit(Xtr, ytr)                                   # only to create the object; weights replaced below
    sk.coef_ = sum(n / N * c for n, c, _ in loc).reshape(1, -1)
    sk.intercept_ = np.array([sum(n / N * i for n, _, i in loc)])

    p_plain = col(sk.predict_proba(Xte))
    rows.append(dict(seed=seed, model="plaintext", **metrics(yte, p_plain, p_plain)))

    for nb in N_BITS_LIST:
        cm = ConcreteLR.from_sklearn_model(sk, Xtr, n_bits=nb)
        cm.compile(Xtr)
        p_enc = col(cm.predict_proba(Xte, fhe="execute"))      # real encrypted execution
        r = metrics(yte, p_enc, p_plain)
        r["label_changes"] = int(np.sum((p_enc >= .5) != (p_plain >= .5)))   # patients whose label differs from plaintext
        rows.append(dict(seed=seed, model=f"encrypted_{nb}bit", **r))
    print(f"seed {seed} done  (test patients: {len(yte)})")

res = pd.DataFrame(rows)
res.to_csv("utility_results.csv", index=False)

cols = ["accuracy", "precision", "recall", "f1", "error_rel", "mse"]
print(f"\n=== mean ± std over {N_SEEDS} splits ===")
print(f"{'model':<18}" + "".join(f"{c:>16}" for c in cols))
for m, g in res.groupby("model", sort=False):
    print(f"{m:<18}" + "".join(f"{g[c].mean():>9.4f}±{g[c].std():.4f}" for c in cols))

print("\n=== encrypted minus plaintext (paired by split) ===")
plain = res[res.model == "plaintext"].set_index("seed")
for m in [f"encrypted_{nb}bit" for nb in N_BITS_LIST]:
    g = res[res.model == m].set_index("seed")
    d = (g["accuracy"] - plain["accuracy"])
    print(f"{m}: accuracy difference = {d.mean():+.4f} ± {d.std():.4f} ; "
          f"total patients whose label changed = {int(g['label_changes'].sum())} "
          f"out of {N_SEEDS * len(yte)} predictions")
print("\nSaved utility_results.csv")

Writing eval_utility.py


In [4]:
!/content/cml/bin/python eval_utility.py 2>&1 | grep -v -i warn

seed 0 done  (test patients: 192)
seed 1 done  (test patients: 192)
seed 2 done  (test patients: 192)
seed 3 done  (test patients: 192)
seed 4 done  (test patients: 192)
seed 5 done  (test patients: 192)
seed 6 done  (test patients: 192)
seed 7 done  (test patients: 192)
seed 8 done  (test patients: 192)
seed 9 done  (test patients: 192)

=== mean ± std over 10 splits ===
model                     accuracy       precision          recall              f1       error_rel             mse
plaintext            0.7594±0.0225   0.7045±0.0489   0.5403±0.0544   0.6096±0.0403   0.0000±0.0000   0.1633±0.0128
encrypted_6bit       0.7599±0.0274   0.7075±0.0585   0.5373±0.0558   0.6090±0.0471   0.0306±0.0028   0.1628±0.0128
encrypted_8bit       0.7573±0.0219   0.7032±0.0481   0.5328±0.0563   0.6041±0.0407   0.0073±0.0008   0.1632±0.0128

=== encrypted minus plaintext (paired by split) ===
encrypted_6bit: accuracy difference = +0.0005 ± 0.0071 ; total patients whose label changed = 23 out of 1920 pre

In [5]:
from google.colab import files
files.download("utility_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
%%writefile eval_performance.py
import sys, os, json, time, subprocess, platform, threading, tempfile
import numpy as np, pandas as pd

# ---- settings (edit these) ----
CSV, TARGET = "pima.csv", "Outcome"
K = 5                       # number of hospitals
N_SEEDS = 5                 # repeated runs (each in a fresh process, so memory is clean)
N_BITS_LIST = [6, 8]
N_ENC = 300                 # encrypted samples timed per run
MIN_PLAIN_SECONDS = 2.0     # plaintext loops run at least this long (so CPU% can be sampled)


def prep(seed):
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression as SkLR
    df = pd.read_csv(CSV)
    y = df[TARGET].astype(int).values
    for c in ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]:
        if c in df.columns:
            df[c] = df[c].replace(0, np.nan)
    X = df.drop(columns=[TARGET]).astype(float).values
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=seed)
    imp = SimpleImputer(strategy="median").fit(Xtr)
    sc = StandardScaler().fit(imp.transform(Xtr))
    Xtr, Xte = sc.transform(imp.transform(Xtr)), sc.transform(imp.transform(Xte))
    parts = np.array_split(np.random.default_rng(seed).permutation(len(ytr)), K)
    loc = [(len(i), *(lambda m: (m.coef_[0], m.intercept_[0]))(SkLR(max_iter=2000).fit(Xtr[i], ytr[i]))) for i in parts]
    N = sum(n for n, _, _ in loc)
    sk = SkLR().fit(Xtr, ytr)
    sk.coef_ = sum(n / N * c for n, c, _ in loc).reshape(1, -1)
    sk.intercept_ = np.array([sum(n / N * i for n, _, i in loc)])
    return Xtr, Xte, ytr, yte, sk


class Sampler:
    """Samples this process' CPU% and RSS every 50 ms in a background thread."""
    def __init__(self, interval=0.05):
        import psutil
        self.p, self.interval, self.cpu, self.rss = psutil.Process(), interval, [], []
        self._stop = threading.Event()
    def __enter__(self):
        self.p.cpu_percent(None)
        self.t = threading.Thread(target=self._run, daemon=True); self.t.start(); return self
    def _run(self):
        while not self._stop.is_set():
            time.sleep(self.interval)
            self.cpu.append(self.p.cpu_percent(None)); self.rss.append(self.p.memory_info().rss)
    def __exit__(self, *a):
        self._stop.set(); self.t.join()
    @property
    def mean_cpu(self): return float(np.mean(self.cpu)) if self.cpu else float("nan")
    @property
    def peak_rss_mb(self): return max(self.rss) / 2**20 if self.rss else float("nan")


def nbytes(o):
    if o is None: return 0
    if isinstance(o, (bytes, bytearray)): return len(o)
    if isinstance(o, (tuple, list)): return sum(nbytes(x) for x in o)
    return len(bytes(o))


def worker(seed, nb):
    import psutil
    Xtr, Xte, ytr, yte, sk = prep(seed)
    proc = psutil.Process()
    rss = lambda: proc.memory_info().rss / 2**20
    out = dict(seed=seed, n_bits=nb, cpu_count=os.cpu_count())
    out["rss_start_mb"] = rss()                       # python + sklearn + data, before any encryption code

    # ---------- plaintext baselines ----------
    def plain_loop(fn, label):
        for i in range(50): fn(Xte[i % len(Xte)].reshape(1, -1))      # warm-up
        ts, i, t_end = [], 0, time.perf_counter() + MIN_PLAIN_SECONDS
        c0 = time.process_time()
        with Sampler() as s:
            while time.perf_counter() < t_end:
                x = Xte[i % len(Xte)].reshape(1, -1); t = time.perf_counter(); fn(x)
                ts.append(time.perf_counter() - t); i += 1
        out[f"{label}_cpu_ms_per_call"] = (time.process_time() - c0) / len(ts) * 1e3   # CPU time, all threads
        out[f"{label}_latency_ms"] = float(np.mean(ts) * 1e3)
        out[f"{label}_cpu_pct"] = s.mean_cpu
        out[f"{label}_rss_peak_mb"] = s.peak_rss_mb
    plain_loop(lambda x: sk.predict_proba(x), "plain_sklearn")
    w, b = sk.coef_[0], sk.intercept_[0]
    plain_loop(lambda x: 1 / (1 + np.exp(-(x @ w + b))), "plain_numpy")

    # ---------- encrypted ----------
    from concrete.ml.sklearn import LogisticRegression as ConcreteLR
    from concrete.ml.deployment import FHEModelClient, FHEModelDev, FHEModelServer
    out["rss_after_import_mb"] = rss()
    cm = ConcreteLR.from_sklearn_model(sk, Xtr, n_bits=nb)
    t = time.perf_counter(); cm.compile(Xtr); out["compile_s"] = time.perf_counter() - t
    out["rss_after_compile_mb"] = rss()
    st = cm.fhe_circuit.statistics
    out["bootstrap_count"] = st.get("programmable_bootstrap_count")
    out["keyswitch_count"] = st.get("key_switch_count")

    with tempfile.TemporaryDirectory() as tmp:
        dev, keys = os.path.join(tmp, "dev"), os.path.join(tmp, "keys")
        FHEModelDev(dev, cm).save()
        for z in ("client", "server"):
            p = os.path.join(dev, f"{z}.zip")
            out[f"{z}_zip_kb"] = os.path.getsize(p) / 1024 if os.path.exists(p) else float("nan")
        client, server = FHEModelClient(dev, key_dir=keys), FHEModelServer(dev)
        server.load()
        t = time.perf_counter(); client.generate_private_and_evaluation_keys(force=True)
        out["keygen_s"] = time.perf_counter() - t
        ek = client.get_serialized_evaluation_keys()
        out["evaluation_key_kb"] = nbytes(ek) / 1024
        out["rss_after_keygen_mb"] = rss()

        def one(x):
            t0 = time.perf_counter(); enc = client.quantize_encrypt_serialize(x)
            t1 = time.perf_counter(); res = server.run(enc, ek)
            t2 = time.perf_counter(); client.deserialize_decrypt_dequantize(res)
            t3 = time.perf_counter()
            return t1 - t0, t2 - t1, t3 - t2, nbytes(enc), nbytes(res)
        for i in range(5): one(Xte[i].reshape(1, -1))                 # warm-up
        r = []
        c0 = time.process_time()
        with Sampler() as s:
            for i in range(N_ENC): r.append(one(Xte[i % len(Xte)].reshape(1, -1)))
        out["enc_cpu_ms_per_call"] = (time.process_time() - c0) / N_ENC * 1e3
        r = np.array(r)
        out["enc_encrypt_ms"], out["enc_server_ms"], out["enc_decrypt_ms"] = (r[:, :3].mean(0) * 1e3).tolist()
        out["enc_total_ms"] = float(r[:, :3].sum(1).mean() * 1e3)
        out["enc_total_ms_std"] = float(r[:, :3].sum(1).std() * 1e3)
        out["ciphertext_in_bytes"], out["ciphertext_out_bytes"] = float(r[:, 3].mean()), float(r[:, 4].mean())
        out["enc_cpu_pct"], out["enc_rss_peak_mb"] = s.mean_cpu, s.peak_rss_mb

    # ---------- derived ----------
    out["overhead_total_vs_sklearn"] = out["enc_total_ms"] / out["plain_sklearn_latency_ms"]
    out["overhead_server_vs_sklearn"] = out["enc_server_ms"] / out["plain_sklearn_latency_ms"]
    out["overhead_total_vs_numpy"] = out["enc_total_ms"] / out["plain_numpy_latency_ms"]
    out["plain_in_bytes"], out["plain_out_bytes"] = Xte.shape[1] * 8, 8
    out["fl_update_bytes_per_hospital_per_round"] = (len(w) + 1) * 8   # plaintext FedAvg upload (float64 weights + bias)
    out["rss_increase_mb"] = out["enc_rss_peak_mb"] - out["rss_start_mb"]
    print("RESULT " + json.dumps(out))


def main():
    cpu = "unknown"
    try:
        cpu = [l.split(":")[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")][0]
    except Exception: pass
    print(f"Machine: {cpu}, {os.cpu_count()} logical CPUs, Python {platform.python_version()}")
    rows = []
    for nb in N_BITS_LIST:
        for seed in range(N_SEEDS):
            p = subprocess.run([sys.executable, __file__, "--worker", str(seed), str(nb)], capture_output=True, text=True)
            line = [l for l in p.stdout.splitlines() if l.startswith("RESULT ")]
            if not line:
                print("worker failed:\n", p.stderr[-1500:]); sys.exit(1)
            rows.append(json.loads(line[0][7:])); print(f"  n_bits={nb} seed={seed} done")
    df = pd.DataFrame(rows); df.to_csv("performance_results.csv", index=False)

    def show(title, items):
        print(f"\n=== {title} ===")
        for nb, g in df.groupby("n_bits"):
            print(f"-- {nb}-bit quantization (mean ± std over {len(g)} runs)")
            for label, col, unit in items:
                print(f"   {label:<48}{g[col].mean():>12.4g} ± {g[col].std():<9.2g}{unit}")
    show("Inference latency (per sample)", [
        ("plaintext, scikit-learn predict_proba", "plain_sklearn_latency_ms", "ms"),
        ("plaintext, NumPy dot + sigmoid", "plain_numpy_latency_ms", "ms"),
        ("encrypted: client encrypt", "enc_encrypt_ms", "ms"),
        ("encrypted: server evaluation", "enc_server_ms", "ms"),
        ("encrypted: client decrypt", "enc_decrypt_ms", "ms"),
        ("encrypted: TOTAL", "enc_total_ms", "ms"),
        ("one-time: key generation", "keygen_s", "s"),
        ("one-time: compilation", "compile_s", "s")])
    show("Computational overhead (encrypted / plaintext)", [
        ("total encrypted vs scikit-learn", "overhead_total_vs_sklearn", "x"),
        ("server-only vs scikit-learn", "overhead_server_vs_sklearn", "x"),
        ("total encrypted vs NumPy", "overhead_total_vs_numpy", "x")])
    show("Communication cost", [
        ("encrypted input per sample (client->server)", "ciphertext_in_bytes", "bytes"),
        ("encrypted output per sample (server->client)", "ciphertext_out_bytes", "bytes"),
        ("plaintext input per sample (8 floats)", "plain_in_bytes", "bytes"),
        ("evaluation keys (one-time)", "evaluation_key_kb", "KB"),
        ("client.zip (one-time)", "client_zip_kb", "KB"),
        ("server.zip (one-time)", "server_zip_kb", "KB"),
        ("FedAvg update per hospital per round (plaintext)", "fl_update_bytes_per_hospital_per_round", "bytes")])
    show("CPU usage (this process, % of ONE core; can exceed 100)", [
        ("plaintext scikit-learn", "plain_sklearn_cpu_pct", "%"),
        ("plaintext NumPy", "plain_numpy_cpu_pct", "%"),
        ("encrypted inference", "enc_cpu_pct", "%")])
    show("CPU time per inference (CPU-seconds used by the process, all threads)", [
        ("plaintext scikit-learn", "plain_sklearn_cpu_ms_per_call", "ms"),
        ("plaintext NumPy", "plain_numpy_cpu_ms_per_call", "ms"),
        ("encrypted (client + server)", "enc_cpu_ms_per_call", "ms")])
    show("Memory use (process RSS)", [
        ("before encryption code (python+sklearn+data)", "rss_start_mb", "MB"),
        ("after importing Concrete ML", "rss_after_import_mb", "MB"),
        ("after compilation", "rss_after_compile_mb", "MB"),
        ("peak during encrypted inference", "enc_rss_peak_mb", "MB"),
        ("increase vs. before encryption code", "rss_increase_mb", "MB")])
    print(f"\nBootstrap operations in circuit: {df.bootstrap_count.iloc[0]}, key-switches: {df.keyswitch_count.iloc[0]}")
    print("Saved performance_results.csv")


if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "--worker":
        worker(int(sys.argv[2]), int(sys.argv[3]))
    else:
        main()

Writing eval_performance.py


In [7]:
!/content/cml/bin/python eval_performance.py

Machine: Intel(R) Xeon(R) CPU @ 2.20GHz, 2 logical CPUs, Python 3.11.17
  n_bits=6 seed=0 done
  n_bits=6 seed=1 done
  n_bits=6 seed=2 done
  n_bits=6 seed=3 done
  n_bits=6 seed=4 done
  n_bits=8 seed=0 done
  n_bits=8 seed=1 done
  n_bits=8 seed=2 done
  n_bits=8 seed=3 done
  n_bits=8 seed=4 done

=== Inference latency (per sample) ===
-- 6-bit quantization (mean ± std over 5 runs)
   plaintext, scikit-learn predict_proba                  0.188 ± 0.081    ms
   plaintext, NumPy dot + sigmoid                       0.01326 ± 0.011    ms
   encrypted: client encrypt                              6.462 ± 2.5      ms
   encrypted: server evaluation                           6.307 ± 3        ms
   encrypted: client decrypt                              3.147 ± 1.2      ms
   encrypted: TOTAL                                       15.92 ± 6.6      ms
   one-time: key generation                            0.001435 ± 0.00038  s
   one-time: compilation                                 0.5237 ± 

In [8]:
from google.colab import files
files.download("performance_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
!ls /sys/class/powercap 2>&1 | head

In [10]:
import os, tempfile, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression as SkLR
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
from concrete.ml.sklearn import LogisticRegression as ConcreteLR
from concrete.ml.deployment import FHEModelClient, FHEModelDev
warnings.filterwarnings("ignore")

# ---- settings (edit these) ----
CSV, TARGET = "pima.csv", "Outcome"
K = 5                      # hospitals
N_SEEDS = 10               # repeated splits
N_BITS_LIST = [6, 8]
COVERAGE = 1.0             # share of the population that appears in the attacker's registry (1.0 = worst case)
PERF_CSV = "performance_results.csv"   # optional: used for the PU score overhead
LEAK_FLOOR = 0.01          # leakage values are clipped at this floor in the PU score (avoids division by zero)

df = pd.read_csv(CSV)
y_all = df[TARGET].astype(int).values
X_raw = df.drop(columns=[TARGET])
X_all = X_raw.copy()
for c in ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]:
    if c in X_all.columns:
        X_all[c] = X_all[c].replace(0, np.nan)
X_all = X_all.astype(float).values
col = lambda p: p[:, 1] if p.ndim == 2 else p

QI_SETS = {   # quasi-identifier sets the attacker links on (computed from the RAW columns)
    "Age(5y band)+Pregnancies": lambda d: list(zip(d["Age"] // 5, d["Pregnancies"])),
    "Age+Pregnancies": lambda d: list(zip(d["Age"], d["Pregnancies"])),
    "Age+Pregnancies+BMI(int)": lambda d: list(zip(d["Age"], d["Pregnancies"], d["BMI"].round().astype(int))),
}

def prep(seed):
    idx = np.arange(len(y_all))
    Xtr, Xte, ytr, yte, itr, ite = train_test_split(X_all, y_all, idx, test_size=0.25, stratify=y_all, random_state=seed)
    imp = SimpleImputer(strategy="median").fit(Xtr)
    sc = StandardScaler().fit(imp.transform(Xtr))
    Xtr, Xte = sc.transform(imp.transform(Xtr)), sc.transform(imp.transform(Xte))
    parts = np.array_split(np.random.default_rng(seed).permutation(len(ytr)), K)
    loc = [(len(i), *(lambda m: (m.coef_[0], m.intercept_[0]))(SkLR(max_iter=2000).fit(Xtr[i], ytr[i]))) for i in parts]
    N = sum(n for n, _, _ in loc)
    sk = SkLR().fit(Xtr, ytr)
    sk.coef_ = sum(n / N * c for n, c, _ in loc).reshape(1, -1)
    sk.intercept_ = np.array([sum(n / N * i for n, _, i in loc)])
    return Xtr, Xte, ytr, yte, itr, ite, sk

def attack(s_mem, s_non, seed):
    """Threshold attack. Score = attacker's membership score (higher = more likely member).
    Members/non-members balanced; threshold chosen on a calibration half, reported on the other half."""
    rng = np.random.default_rng(seed)
    n = min(len(s_mem), len(s_non))
    s = np.concatenate([s_mem[rng.choice(len(s_mem), n, replace=False)], s_non[rng.choice(len(s_non), n, replace=False)]])
    l = np.r_[np.ones(n), np.zeros(n)].astype(int)
    auc = roc_auc_score(l, s)
    fc, tc, _ = roc_curve(l, s)
    s_cal, s_ev, l_cal, l_ev = train_test_split(s, l, test_size=.5, stratify=l, random_state=seed)
    best_t, best = None, -1
    for t in np.unique(s_cal):
        pr = s_cal >= t
        ba = .5 * (pr[l_cal == 1].mean() + 1 - pr[l_cal == 0].mean())
        if ba > best: best, best_t = ba, t
    pr = s_ev >= best_t
    tpr, fpr = pr[l_ev == 1].mean(), pr[l_ev == 0].mean()
    return dict(auc=auc, accuracy=.5 * (tpr + 1 - fpr), advantage=tpr - fpr, tpr_at_1pct_fpr=float(np.interp(0.01, fc, tc)))

true_label_score = lambda p, y: np.where(y == 1, p, 1 - p)     # attacker who knows the true label
label_free_score = lambda p: np.maximum(p, 1 - p)              # attacker who does not (max confidence)

def to_bytes(o):
    return b"".join(o) if isinstance(o, (tuple, list)) else bytes(o)

def ciphertext_attack(cm, Xtr, Xte, seed):
    """Cloud-side attacker that only sees ciphertexts: random forest on raw ciphertext bytes."""
    rng = np.random.default_rng(seed)
    n = len(Xte)
    mem = Xtr[rng.choice(len(Xtr), n, replace=False)]
    with tempfile.TemporaryDirectory() as tmp:
        dev, keys = os.path.join(tmp, "dev"), os.path.join(tmp, "keys")
        FHEModelDev(dev, cm).save()
        client = FHEModelClient(dev, key_dir=keys)
        client.generate_private_and_evaluation_keys(force=True)
        enc = lambda x: np.frombuffer(to_bytes(client.quantize_encrypt_serialize(x.reshape(1, -1))), dtype=np.uint8)
        C = [enc(x) for x in mem] + [enc(x) for x in Xte]
        a, b = enc(Xte[0]), enc(Xte[0])
        identical = float(len(a) == len(b) and np.array_equal(a, b))
    L = min(len(c) for c in C)
    Xc = np.array([c[:L] for c in C], dtype=np.float32) / 255.0
    yc = np.r_[np.ones(n), np.zeros(n)].astype(int)
    Xa, Xb, ya, yb = train_test_split(Xc, yc, test_size=.5, stratify=yc, random_state=seed)
    rf = RandomForestClassifier(200, random_state=seed, n_jobs=-1).fit(Xa, ya)
    p = rf.predict_proba(Xb)[:, 1]
    pr = p >= .5
    tpr, fpr = pr[yb == 1].mean(), pr[yb == 0].mean()
    return dict(auc=roc_auc_score(yb, p), accuracy=.5 * (tpr + 1 - fpr), advantage=tpr - fpr), identical

def reidentify(ite, seed):
    """Linkage attack by the cloud on queried (test) patients, if it sees plaintext quasi-identifiers."""
    rng = np.random.default_rng(seed)
    reg = np.sort(rng.permutation(len(df))[: int(COVERAGE * len(df))])
    in_reg = np.zeros(len(df), bool); in_reg[reg] = True
    out = {}
    for name, fn in QI_SETS.items():
        keys = pd.Series(fn(df))
        cnt = keys.iloc[reg].value_counts()
        k = np.array([cnt.get(keys.iloc[i], 0) for i in ite])
        ok = in_reg[ite] & (k > 0)
        out[name] = (np.mean(np.where(ok, 1.0 / np.maximum(k, 1), 0.0)),     # expected re-identification probability
                     np.mean(ok & (k == 1)))                                  # share uniquely matched
    return out, 1.0 / len(reg)

rows = []
add = lambda seed, scenario, model, metric, v: rows.append(dict(seed=seed, scenario=scenario, model=model, metric=metric, value=float(v)))

for seed in range(N_SEEDS):
    Xtr, Xte, ytr, yte, itr, ite, sk = prep(seed)
    ptr, pte = col(sk.predict_proba(Xtr)), col(sk.predict_proba(Xte))
    add(seed, "utility", "plaintext", "accuracy", accuracy_score(yte, pte >= .5))
    # A: output-access attacker (receives the prediction and knows the true label)
    for k, v in attack(true_label_score(ptr, ytr), true_label_score(pte, yte), seed).items():
        add(seed, "A_output_access", "plaintext", k, v)
    # B1: cloud sees plaintext queries -> label-free confidence attack
    for k, v in attack(label_free_score(ptr), label_free_score(pte), seed).items():
        add(seed, "B_cloud", "plaintext", k, v)
    # C: re-identification by the cloud with plaintext quasi-identifiers
    reid, chance = reidentify(ite, seed)
    for name, (p_exp, p_uni) in reid.items():
        add(seed, "C_reid_" + name, "plaintext", "reid_expected", p_exp)
        add(seed, "C_reid_" + name, "plaintext", "reid_unique", p_uni)
        add(seed, "C_reid_" + name, "pulse (ciphertext only)", "reid_expected", chance)

    for nb in N_BITS_LIST:
        m = f"pulse_{nb}bit"
        cm = ConcreteLR.from_sklearn_model(sk, Xtr, n_bits=nb); cm.compile(Xtr)
        etr, ete = col(cm.predict_proba(Xtr, fhe="execute")), col(cm.predict_proba(Xte, fhe="execute"))
        add(seed, "utility", m, "accuracy", accuracy_score(yte, ete >= .5))
        for k, v in attack(true_label_score(etr, ytr), true_label_score(ete, yte), seed).items():
            add(seed, "A_output_access", m, k, v)
        res, identical = ciphertext_attack(cm, Xtr, Xte, seed)
        for k, v in res.items():
            add(seed, "B_cloud", m + " (ciphertext only)", k, v)
        add(seed, "B_cloud", m + " (ciphertext only)", "same_input_gives_identical_ciphertext", identical)
    print(f"seed {seed} done")

R = pd.DataFrame(rows); R.to_csv("privacy_results.csv", index=False)
g = R.groupby(["scenario", "model", "metric"], sort=False)["value"].agg(["mean", "std"]).reset_index()

def show(title, scen, metrics):
    print(f"\n=== {title} ===")
    sub = g[(g.scenario == scen) & (g.metric.isin(metrics))]
    for (mod), s in sub.groupby("model", sort=False):
        print(f"  {mod:<34}" + "  ".join(f"{r.metric}={r['mean']:.3f}±{r['std']:.3f}" for _, r in s.iterrows()))

M4 = ["auc", "accuracy", "advantage", "tpr_at_1pct_fpr"]
show("A. MIA, attacker who receives the (decrypted) prediction and knows the true label", "A_output_access", M4)
show("B. MIA, cloud attacker (plaintext hosting: max-confidence score; PULSE: ciphertext bytes only)", "B_cloud", M4 + ["same_input_gives_identical_ciphertext"])
print("\n=== C. Re-identification by the cloud (linkage on quasi-identifiers; registry covers "
      f"{COVERAGE:.0%} of patients; {len(ite)} queried patients) ===")
for scen in [s for s in g.scenario.unique() if s.startswith("C_")]:
    s = g[g.scenario == scen]
    pl = s[(s.model == "plaintext") & (s.metric == "reid_expected")].iloc[0]
    un = s[(s.model == "plaintext") & (s.metric == "reid_unique")].iloc[0]
    pu = s[(s.model != "plaintext")].iloc[0]
    print(f"  {scen[7:]:<28} plaintext: expected {pl['mean']*100:5.1f}% (uniquely matched {un['mean']*100:5.1f}%)   "
          f"PULSE: chance level {pu['mean']*100:.2f}% (not measured)")

# ---- PU score (Eq. 15: accuracy / (overhead x leakage)), leakage clipped at LEAK_FLOOR ----
if os.path.exists(PERF_CSV):
    perf = pd.read_csv(PERF_CSV).groupby("n_bits")["overhead_total_vs_sklearn"].mean()
    mean = lambda scen, model, metric: g[(g.scenario == scen) & (g.model == model) & (g.metric == metric)]["mean"].iloc[0]
    clip = lambda v: max(v, LEAK_FLOOR)
    print(f"\n=== PU score = accuracy / (overhead x leakage); leakage = MIA advantage clipped at {LEAK_FLOOR} ===")
    print("  (a) leakage = output-access attacker advantage   (b) leakage = cloud attacker advantage")
    for model, oh, ca in [("plaintext", 1.0, "plaintext")] + [(f"pulse_{nb}bit", perf.get(nb, np.nan), f"pulse_{nb}bit (ciphertext only)") for nb in N_BITS_LIST]:
        acc = mean("utility", model, "accuracy")
        la, lb = clip(mean("A_output_access", model, "advantage")), clip(mean("B_cloud", ca, "advantage"))
        print(f"  {model:<14} accuracy={acc:.3f} overhead={oh:6.1f}x  (a) leak={la:.3f} PU={acc/(oh*la):7.2f}   (b) leak={lb:.3f} PU={acc/(oh*lb):7.2f}")
else:
    print(f"\n({PERF_CSV} not found, PU score skipped: upload it to compute the overhead term)")
print("\nSaved privacy_results.csv")

ModuleNotFoundError: No module named 'concrete'

In [11]:
%%writefile privacy_eval.py

Writing privacy_eval.py


In [12]:
!ls
!ls /content/cml/bin/python

cml		     performance_results.csv  sample_data
eval_performance.py  pima.csv		      utility_results.csv
eval_utility.py      privacy_eval.py
/content/cml/bin/python


In [13]:
!/content/cml/bin/python privacy_eval.py 2>&1 | grep -v -i -E "warn|KeySetCache"

In [14]:
!wc -l privacy_eval.py
!head -3 privacy_eval.py
!tail -3 privacy_eval.py

0 privacy_eval.py
  

In [15]:
%%writefile privacy_eval.py
import os, tempfile, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression as SkLR
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
from concrete.ml.sklearn import LogisticRegression as ConcreteLR
from concrete.ml.deployment import FHEModelClient, FHEModelDev
warnings.filterwarnings("ignore")

# ---- settings (edit these) ----
CSV, TARGET = "pima.csv", "Outcome"
K = 5                      # hospitals
N_SEEDS = 10               # repeated splits
N_BITS_LIST = [6, 8]
COVERAGE = 1.0             # share of the population that appears in the attacker's registry (1.0 = worst case)
PERF_CSV = "performance_results.csv"   # optional: used for the PU score overhead
LEAK_FLOOR = 0.01          # leakage values are clipped at this floor in the PU score (avoids division by zero)

df = pd.read_csv(CSV)
y_all = df[TARGET].astype(int).values
X_raw = df.drop(columns=[TARGET])
X_all = X_raw.copy()
for c in ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]:
    if c in X_all.columns:
        X_all[c] = X_all[c].replace(0, np.nan)
X_all = X_all.astype(float).values
col = lambda p: p[:, 1] if p.ndim == 2 else p

QI_SETS = {   # quasi-identifier sets the attacker links on (computed from the RAW columns)
    "Age(5y band)+Pregnancies": lambda d: list(zip(d["Age"] // 5, d["Pregnancies"])),
    "Age+Pregnancies": lambda d: list(zip(d["Age"], d["Pregnancies"])),
    "Age+Pregnancies+BMI(int)": lambda d: list(zip(d["Age"], d["Pregnancies"], d["BMI"].round().astype(int))),
}

def prep(seed):
    idx = np.arange(len(y_all))
    Xtr, Xte, ytr, yte, itr, ite = train_test_split(X_all, y_all, idx, test_size=0.25, stratify=y_all, random_state=seed)
    imp = SimpleImputer(strategy="median").fit(Xtr)
    sc = StandardScaler().fit(imp.transform(Xtr))
    Xtr, Xte = sc.transform(imp.transform(Xtr)), sc.transform(imp.transform(Xte))
    parts = np.array_split(np.random.default_rng(seed).permutation(len(ytr)), K)
    loc = [(len(i), *(lambda m: (m.coef_[0], m.intercept_[0]))(SkLR(max_iter=2000).fit(Xtr[i], ytr[i]))) for i in parts]
    N = sum(n for n, _, _ in loc)
    sk = SkLR().fit(Xtr, ytr)
    sk.coef_ = sum(n / N * c for n, c, _ in loc).reshape(1, -1)
    sk.intercept_ = np.array([sum(n / N * i for n, _, i in loc)])
    return Xtr, Xte, ytr, yte, itr, ite, sk

def attack(s_mem, s_non, seed):
    """Threshold attack. Score = attacker's membership score (higher = more likely member).
    Members/non-members balanced; threshold chosen on a calibration half, reported on the other half."""
    rng = np.random.default_rng(seed)
    n = min(len(s_mem), len(s_non))
    s = np.concatenate([s_mem[rng.choice(len(s_mem), n, replace=False)], s_non[rng.choice(len(s_non), n, replace=False)]])
    l = np.r_[np.ones(n), np.zeros(n)].astype(int)
    auc = roc_auc_score(l, s)
    fc, tc, _ = roc_curve(l, s)
    s_cal, s_ev, l_cal, l_ev = train_test_split(s, l, test_size=.5, stratify=l, random_state=seed)
    best_t, best = None, -1
    for t in np.unique(s_cal):
        pr = s_cal >= t
        ba = .5 * (pr[l_cal == 1].mean() + 1 - pr[l_cal == 0].mean())
        if ba > best: best, best_t = ba, t
    pr = s_ev >= best_t
    tpr, fpr = pr[l_ev == 1].mean(), pr[l_ev == 0].mean()
    return dict(auc=auc, accuracy=.5 * (tpr + 1 - fpr), advantage=tpr - fpr, tpr_at_1pct_fpr=float(np.interp(0.01, fc, tc)))

true_label_score = lambda p, y: np.where(y == 1, p, 1 - p)     # attacker who knows the true label
label_free_score = lambda p: np.maximum(p, 1 - p)              # attacker who does not (max confidence)

def to_bytes(o):
    return b"".join(o) if isinstance(o, (tuple, list)) else bytes(o)

def ciphertext_attack(cm, Xtr, Xte, seed):
    """Cloud-side attacker that only sees ciphertexts: random forest on raw ciphertext bytes."""
    rng = np.random.default_rng(seed)
    n = len(Xte)
    mem = Xtr[rng.choice(len(Xtr), n, replace=False)]
    with tempfile.TemporaryDirectory() as tmp:
        dev, keys = os.path.join(tmp, "dev"), os.path.join(tmp, "keys")
        FHEModelDev(dev, cm).save()
        client = FHEModelClient(dev, key_dir=keys)
        client.generate_private_and_evaluation_keys(force=True)
        enc = lambda x: np.frombuffer(to_bytes(client.quantize_encrypt_serialize(x.reshape(1, -1))), dtype=np.uint8)
        C = [enc(x) for x in mem] + [enc(x) for x in Xte]
        a, b = enc(Xte[0]), enc(Xte[0])
        identical = float(len(a) == len(b) and np.array_equal(a, b))
    L = min(len(c) for c in C)
    Xc = np.array([c[:L] for c in C], dtype=np.float32) / 255.0
    yc = np.r_[np.ones(n), np.zeros(n)].astype(int)
    Xa, Xb, ya, yb = train_test_split(Xc, yc, test_size=.5, stratify=yc, random_state=seed)
    rf = RandomForestClassifier(200, random_state=seed, n_jobs=-1).fit(Xa, ya)
    p = rf.predict_proba(Xb)[:, 1]
    pr = p >= .5
    tpr, fpr = pr[yb == 1].mean(), pr[yb == 0].mean()
    return dict(auc=roc_auc_score(yb, p), accuracy=.5 * (tpr + 1 - fpr), advantage=tpr - fpr), identical

def reidentify(ite, seed):
    """Linkage attack by the cloud on queried (test) patients, if it sees plaintext quasi-identifiers."""
    rng = np.random.default_rng(seed)
    reg = np.sort(rng.permutation(len(df))[: int(COVERAGE * len(df))])
    in_reg = np.zeros(len(df), bool); in_reg[reg] = True
    out = {}
    for name, fn in QI_SETS.items():
        keys = pd.Series(fn(df))
        cnt = keys.iloc[reg].value_counts()
        k = np.array([cnt.get(keys.iloc[i], 0) for i in ite])
        ok = in_reg[ite] & (k > 0)
        out[name] = (np.mean(np.where(ok, 1.0 / np.maximum(k, 1), 0.0)),     # expected re-identification probability
                     np.mean(ok & (k == 1)))                                  # share uniquely matched
    return out, 1.0 / len(reg)

rows = []
add = lambda seed, scenario, model, metric, v: rows.append(dict(seed=seed, scenario=scenario, model=model, metric=metric, value=float(v)))

for seed in range(N_SEEDS):
    Xtr, Xte, ytr, yte, itr, ite, sk = prep(seed)
    ptr, pte = col(sk.predict_proba(Xtr)), col(sk.predict_proba(Xte))
    add(seed, "utility", "plaintext", "accuracy", accuracy_score(yte, pte >= .5))
    # A: output-access attacker (receives the prediction and knows the true label)
    for k, v in attack(true_label_score(ptr, ytr), true_label_score(pte, yte), seed).items():
        add(seed, "A_output_access", "plaintext", k, v)
    # B1: cloud sees plaintext queries -> label-free confidence attack
    for k, v in attack(label_free_score(ptr), label_free_score(pte), seed).items():
        add(seed, "B_cloud", "plaintext", k, v)
    # C: re-identification by the cloud with plaintext quasi-identifiers
    reid, chance = reidentify(ite, seed)
    for name, (p_exp, p_uni) in reid.items():
        add(seed, "C_reid_" + name, "plaintext", "reid_expected", p_exp)
        add(seed, "C_reid_" + name, "plaintext", "reid_unique", p_uni)
        add(seed, "C_reid_" + name, "pulse (ciphertext only)", "reid_expected", chance)

    for nb in N_BITS_LIST:
        m = f"pulse_{nb}bit"
        cm = ConcreteLR.from_sklearn_model(sk, Xtr, n_bits=nb); cm.compile(Xtr)
        etr, ete = col(cm.predict_proba(Xtr, fhe="execute")), col(cm.predict_proba(Xte, fhe="execute"))
        add(seed, "utility", m, "accuracy", accuracy_score(yte, ete >= .5))
        for k, v in attack(true_label_score(etr, ytr), true_label_score(ete, yte), seed).items():
            add(seed, "A_output_access", m, k, v)
        res, identical = ciphertext_attack(cm, Xtr, Xte, seed)
        for k, v in res.items():
            add(seed, "B_cloud", m + " (ciphertext only)", k, v)
        add(seed, "B_cloud", m + " (ciphertext only)", "same_input_gives_identical_ciphertext", identical)
    print(f"seed {seed} done")

R = pd.DataFrame(rows); R.to_csv("privacy_results.csv", index=False)
g = R.groupby(["scenario", "model", "metric"], sort=False)["value"].agg(["mean", "std"]).reset_index()

def show(title, scen, metrics):
    print(f"\n=== {title} ===")
    sub = g[(g.scenario == scen) & (g.metric.isin(metrics))]
    for (mod), s in sub.groupby("model", sort=False):
        print(f"  {mod:<34}" + "  ".join(f"{r.metric}={r['mean']:.3f}±{r['std']:.3f}" for _, r in s.iterrows()))

M4 = ["auc", "accuracy", "advantage", "tpr_at_1pct_fpr"]
show("A. MIA, attacker who receives the (decrypted) prediction and knows the true label", "A_output_access", M4)
show("B. MIA, cloud attacker (plaintext hosting: max-confidence score; PULSE: ciphertext bytes only)", "B_cloud", M4 + ["same_input_gives_identical_ciphertext"])
print("\n=== C. Re-identification by the cloud (linkage on quasi-identifiers; registry covers "
      f"{COVERAGE:.0%} of patients; {len(ite)} queried patients) ===")
for scen in [s for s in g.scenario.unique() if s.startswith("C_")]:
    s = g[g.scenario == scen]
    pl = s[(s.model == "plaintext") & (s.metric == "reid_expected")].iloc[0]
    un = s[(s.model == "plaintext") & (s.metric == "reid_unique")].iloc[0]
    pu = s[(s.model != "plaintext")].iloc[0]
    print(f"  {scen[7:]:<28} plaintext: expected {pl['mean']*100:5.1f}% (uniquely matched {un['mean']*100:5.1f}%)   "
          f"PULSE: chance level {pu['mean']*100:.2f}% (not measured)")

# ---- PU score (Eq. 15: accuracy / (overhead x leakage)), leakage clipped at LEAK_FLOOR ----
if os.path.exists(PERF_CSV):
    perf = pd.read_csv(PERF_CSV).groupby("n_bits")["overhead_total_vs_sklearn"].mean()
    mean = lambda scen, model, metric: g[(g.scenario == scen) & (g.model == model) & (g.metric == metric)]["mean"].iloc[0]
    clip = lambda v: max(v, LEAK_FLOOR)
    print(f"\n=== PU score = accuracy / (overhead x leakage); leakage = MIA advantage clipped at {LEAK_FLOOR} ===")
    print("  (a) leakage = output-access attacker advantage   (b) leakage = cloud attacker advantage")
    for model, oh, ca in [("plaintext", 1.0, "plaintext")] + [(f"pulse_{nb}bit", perf.get(nb, np.nan), f"pulse_{nb}bit (ciphertext only)") for nb in N_BITS_LIST]:
        acc = mean("utility", model, "accuracy")
        la, lb = clip(mean("A_output_access", model, "advantage")), clip(mean("B_cloud", ca, "advantage"))
        print(f"  {model:<14} accuracy={acc:.3f} overhead={oh:6.1f}x  (a) leak={la:.3f} PU={acc/(oh*la):7.2f}   (b) leak={lb:.3f} PU={acc/(oh*lb):7.2f}")
else:
    print(f"\n({PERF_CSV} not found, PU score skipped: upload it to compute the overhead term)")
print("\nSaved privacy_results.csv")

Overwriting privacy_eval.py


In [16]:
!wc -l privacy_eval.py
!/content/cml/bin/python -u privacy_eval.py 2>&1 | grep -v -i -E "warn|KeySetCache"

184 privacy_eval.py
seed 0 done
seed 1 done
seed 2 done
seed 3 done
seed 4 done
seed 5 done
seed 6 done
seed 7 done
seed 8 done
seed 9 done

=== A. MIA, attacker who receives the (decrypted) prediction and knows the true label ===
  plaintext                         auc=0.505±0.039  accuracy=0.512±0.029  advantage=0.024±0.057  tpr_at_1pct_fpr=0.016±0.015
  pulse_6bit                        auc=0.504±0.038  accuracy=0.512±0.030  advantage=0.024±0.061  tpr_at_1pct_fpr=0.014±0.014
  pulse_8bit                        auc=0.505±0.039  accuracy=0.517±0.028  advantage=0.033±0.055  tpr_at_1pct_fpr=0.016±0.015

=== B. MIA, cloud attacker (plaintext hosting: max-confidence score; PULSE: ciphertext bytes only) ===
  plaintext                         auc=0.486±0.040  accuracy=0.494±0.042  advantage=-0.011±0.085  tpr_at_1pct_fpr=0.011±0.013
  pulse_6bit (ciphertext only)      auc=0.515±0.046  accuracy=0.509±0.044  advantage=0.018±0.087  same_input_gives_identical_ciphertext=0.000±0.000
  pulse_8bit

In [17]:
import pandas as pd
p = pd.read_csv("performance_results.csv")
print(p.groupby("n_bits")[["overhead_total_vs_sklearn","enc_total_ms","plain_sklearn_latency_ms"]].agg(["mean","std"]).round(2))

       overhead_total_vs_sklearn        enc_total_ms        \
                            mean    std         mean   std   
n_bits                                                       
6                          91.63  48.27        15.92  6.65   
8                          56.82  15.86        10.58  2.26   

       plain_sklearn_latency_ms        
                           mean   std  
n_bits                                 
6                          0.19  0.08  
8                          0.19  0.04  


In [22]:
import pandas as pd
p = pd.read_csv("performance_results.csv")
print(p[["n_bits","seed","enc_total_ms","plain_sklearn_latency_ms","overhead_total_vs_sklearn"]].round(2).to_string())

   n_bits  seed  enc_total_ms  plain_sklearn_latency_ms  overhead_total_vs_sklearn
0       6     0         25.96                      0.15                     176.21
1       6     1         19.31                      0.33                      57.99
2       6     2         12.73                      0.15                      83.58
3       6     3         11.60                      0.15                      75.32
4       6     4          9.97                      0.15                      65.06
5       8     0         12.17                      0.15                      81.25
6       8     1         13.76                      0.23                      60.37
7       8     2          8.91                      0.18                      49.34
8       8     3          9.23                      0.17                      54.67
9       8     4          8.82                      0.23                      38.47


In [26]:
!cp performance_results.csv perf_run1.csv

In [27]:
!/content/cml/bin/python eval_performance.py > run2.log 2>&1
!cp performance_results.csv perf_run2.csv

In [28]:
!/content/cml/bin/python eval_performance.py > run3.log 2>&1
!cp performance_results.csv perf_run3.csv

In [29]:
import pandas as pd, glob
files = sorted(glob.glob("perf_run*.csv"))
P = pd.concat([pd.read_csv(f).assign(run=f) for f in files])
cols = ["enc_total_ms", "plain_sklearn_latency_ms", "overhead_total_vs_sklearn", "enc_cpu_ms_per_call"]
print(files)
print(P.groupby("n_bits")[cols].agg(["median", "min", "max"]).round(2).T)
print(len(P) // 2, "runs per bit-width")

['perf_run1.csv', 'perf_run2.csv', 'perf_run3.csv']
n_bits                                 6      8
enc_total_ms              median    8.67   9.23
                          min       8.24   8.62
                          max      24.55  15.57
plain_sklearn_latency_ms  median    0.15   0.16
                          min       0.14   0.15
                          max       0.22   0.23
overhead_total_vs_sklearn median   56.85  58.00
                          min      40.18  41.76
                          max     160.83  88.00
enc_cpu_ms_per_call       median    9.23   9.74
                          min       8.70   9.17
                          max      21.50  23.98
15 runs per bit-width
